# Esquivar meteorits

En aquesta activitat programarem un joc en què un OVNI esquiva meteorits que cauen del cel. Un controlador de lògica difusa tria automàticament cap a on es mou l’OVNI. Cada meteorit esquivat suma un punt; cada col·lisió resta una vida. La partida acaba quan s’esgoten les tres vides.

El joc es dibuixa amb `ipycanvas`, pensat per a mostrar gràfics animats dins de Jupyter. També es pot executar en Google Colab: la primera cel·la habilita els widgets personalitzats que necessita el canvas. Les imatges del joc han d’estar al costat d’aquest notebook.

## 1. Instal·lació i compatibilitat

In [ ]:
# En Colab, habilitem el gestor necessari per als widgets de tercers.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

%pip install ipycanvas ipywidgets scipy scikit-fuzzy pillow

Si Jupyter demana reiniciar el kernel després de la instal·lació, reinicia’l i executa les cel·les des del principi.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import asyncio
import random
import time

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from ipycanvas import Canvas, hold_canvas
from ipywidgets import Button, HBox, Image, VBox, HTML
from IPython.display import display
from PIL import Image as PILImage

# Localitzem els PNG en el repositori; en Colab els descarreguem si només s'ha pujat el notebook.
NOMS_ASSETS = ("ufo_green.png", "meteor.png")
CANDIDATS = (Path.cwd(), Path.cwd() / "apunts/2.-segon_trimestre/6.-Sistemes Basats en Regles/esquivar",
             *(Path.cwd().parents))
DIRECTORI = next((base for base in CANDIDATS
                  if all((base / nom).is_file() for nom in NOMS_ASSETS)), None)
if DIRECTORI is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            "No es troben ufo_green.png i meteor.png. Col·loca'ls al costat del notebook."
        ) from error
    print("En Colab, puja ara els fitxers ufo_green.png i meteor.png.")
    pujats = files.upload()
    DIRECTORI = Path.cwd()
    if not all(nom in pujats for nom in NOMS_ASSETS):
        raise FileNotFoundError("Cal pujar els dos fitxers PNG indicats.")
    for nom in NOMS_ASSETS:
        (DIRECTORI / nom).write_bytes(pujats[nom])

AMPLE, ALT = 800, 500
ufo_img = Image.from_file(str(DIRECTORI / "ufo_green.png"))
meteor_img = Image.from_file(str(DIRECTORI / "meteor.png"))
ufo_w, ufo_h = PILImage.open(DIRECTORI / "ufo_green.png").size
meteor_w, meteor_h = PILImage.open(DIRECTORI / "meteor.png").size

canvas = Canvas(width=AMPLE, height=ALT)
canvas.layout.width = "min(100%, 800px)"
canvas.layout.height = "auto"

## 2. Controlador de lògica difusa

In [ ]:
# La variable d’entrada és la distància horitzontal entre el meteorit i l’OVNI.
distancia = ctrl.Antecedent(np.linspace(-AMPLE, AMPLE, 161), "distancia")
distancia["molt_esquerra"] = fuzz.trapmf(distancia.universe, [-AMPLE, -AMPLE, -260, -150])
distancia["esquerra"] = fuzz.trimf(distancia.universe, [-260, -120, 0])
distancia["centre"] = fuzz.trimf(distancia.universe, [-100, 0, 100])
distancia["dreta"] = fuzz.trimf(distancia.universe, [0, 120, 260])
distancia["molt_dreta"] = fuzz.trapmf(distancia.universe, [150, 260, AMPLE, AMPLE])

# Una velocitat negativa mou l’OVNI a l’esquerra; una positiva, a la dreta.
velocitat = ctrl.Consequent(np.linspace(-6, 6, 121), "velocitat")
velocitat["esquerra_rapida"] = fuzz.trapmf(velocitat.universe, [-6, -6, -4, -2])
velocitat["esquerra"] = fuzz.trimf(velocitat.universe, [-4, -2, 0])
velocitat["aturat"] = fuzz.trimf(velocitat.universe, [-0.6, 0, 0.6])
velocitat["dreta"] = fuzz.trimf(velocitat.universe, [0, 2, 4])
velocitat["dreta_rapida"] = fuzz.trapmf(velocitat.universe, [2, 4, 6, 6])

regles = [
    ctrl.Rule(distancia["molt_esquerra"], velocitat["dreta_rapida"]),
    ctrl.Rule(distancia["esquerra"], velocitat["dreta"]),
    ctrl.Rule(distancia["centre"], velocitat["dreta_rapida"]),
    ctrl.Rule(distancia["dreta"], velocitat["esquerra"]),
    ctrl.Rule(distancia["molt_dreta"], velocitat["esquerra_rapida"]),
]
controlador = ctrl.ControlSystemSimulation(ctrl.ControlSystem(regles))

def calcular_velocitat(dx):
    """Calcula el moviment lateral per allunyar-se del meteorit més perillós."""
    controlador.input["distancia"] = float(np.clip(dx, -AMPLE, AMPLE))
    controlador.compute()
    return float(controlador.output["velocitat"])

## 3. Estat, dibuix i lògica de la partida

In [ ]:
UFO_Y = ALT - ufo_h / 2 - 12
MARGE_X = ufo_w / 2
estat = {"x": AMPLE / 2, "meteorits": [], "punts": 0, "vides": 3,
         "actiu": False, "tancat": False, "ultim_meteorit": 0.0,
         "vx": 0.0, "tasca": None, "generacio": 0}

estat_text = HTML()


def dibuixar(missatge=""):
    """Redibuixa el fotograma complet en una única transmissió al navegador."""
    with hold_canvas(canvas):
        canvas.clear()
        canvas.fill_style = "#101827"
        canvas.fill_rect(0, 0, AMPLE, ALT)
        # Estrelles fixes: el fons no parpelleja entre fotogrames.
        canvas.fill_style = "#dbeafe"
        for sx, sy, radi in estrelles:
            canvas.fill_circle(sx, sy, radi)
        for m in estat["meteorits"]:
            canvas.draw_image(meteor_img, m["x"] - meteor_w / 2,
                              m["y"] - meteor_h / 2, meteor_w, meteor_h)
        canvas.draw_image(ufo_img, estat["x"] - ufo_w / 2,
                          UFO_Y - ufo_h / 2, ufo_w, ufo_h)
        canvas.fill_style = "white"
        canvas.font = "bold 20px sans-serif"
        canvas.fill_text(f"Punts: {estat['punts']}", 18, 30)
        canvas.fill_style = "#ff7777"
        canvas.fill_text(f"Vides: {estat['vides']}", 18, 58)
        canvas.fill_style = "#b9d8ff"
        canvas.font = "16px sans-serif"
        canvas.fill_text(f"Velocitat: {estat['vx']:+.1f}", 18, 84)
        if missatge:
            canvas.fill_style = "rgba(0, 0, 0, 0.68)"
            canvas.fill_rect(0, 0, AMPLE, ALT)
            canvas.fill_style = "white"
            canvas.text_align = "center"
            canvas.font = "bold 38px sans-serif"
            canvas.fill_text(missatge, AMPLE / 2, ALT / 2)
            canvas.text_align = "left"


random.seed(14)
estrelles = [(random.randrange(AMPLE), random.randrange(ALT), random.choice([1, 1, 2]))
            for _ in range(65)]


def actualitzar():
    """Avança un pas de simulació i resol els esquivaments i les col·lisions."""
    ara = time.monotonic()
    if ara - estat["ultim_meteorit"] >= 0.85:
        estat["meteorits"].append({"x": random.uniform(meteor_w / 2, AMPLE - meteor_w / 2),
                                   "y": -meteor_h / 2})
        estat["ultim_meteorit"] = ara

    amenaces = []
    for m in estat["meteorits"]:
        dy = UFO_Y - m["y"]
        dx = m["x"] - estat["x"]
        if -meteor_h < dy < 210 and abs(dx) < 260:
            # Prioritza els meteorits pròxims a l’OVNI i alineats amb ell.
            amenaces.append((abs(dx) + 0.55 * max(dy, 0), dx))

    if amenaces:
        _, dx = min(amenaces, key=lambda parella: parella[0])
        estat["vx"] = calcular_velocitat(dx)
    else:
        estat["vx"] = 0.0

    nova_x = estat["x"] + estat["vx"]
    # Si la direcció d’evasió està bloquejada per la vora, provem l’altra banda.
    if nova_x < MARGE_X or nova_x > AMPLE - MARGE_X:
        estat["vx"] = -estat["vx"]
        nova_x = estat["x"] + estat["vx"]
    estat["x"] = float(np.clip(nova_x, MARGE_X, AMPLE - MARGE_X))
    sobreviuen = []
    for m in estat["meteorits"]:
        m["y"] += 3.2
        dx = abs(m["x"] - estat["x"])
        dy = abs(m["y"] - UFO_Y)
        col_lisio = dx < (ufo_w + meteor_w) * 0.34 and dy < (ufo_h + meteor_h) * 0.34
        if col_lisio:
            estat["vides"] -= 1
        elif m["y"] - meteor_h / 2 > ALT:
            estat["punts"] += 1
        else:
            sobreviuen.append(m)
    estat["meteorits"] = sobreviuen
    return estat["vides"] > 0

## 4. Controls i inici

In [ ]:
async def bucle_joc(generacio):
    """Bucle asíncron: deixa que Jupyter/Colab continue processant els controls."""
    estat["actiu"] = True
    try:
        while generacio == estat["generacio"] and estat["actiu"] and estat["vides"] > 0:
            continua = actualitzar()
            dibuixar()
            estat_text.value = (
                f"<b>Partida en marxa</b> · Punts: {estat['punts']} · "
                f"Vides: {estat['vides']}"
            )
            if not continua:
                break
            await asyncio.sleep(1 / 30)
    except asyncio.CancelledError:
        pass
    finally:
        if generacio == estat["generacio"]:
            estat["actiu"] = False
        if generacio == estat["generacio"] and estat["vides"] <= 0:
            dibuixar("FI DE LA PARTIDA")
            estat_text.value = f"<b>Fi de la partida.</b> Puntuació: {estat['punts']}"


def iniciar(_=None):
    """Inicia o reinicia el joc i cancel·la qualsevol bucle anterior."""
    tasca = estat.get("tasca")
    if tasca is not None and not tasca.done():
        tasca.cancel()
    estat["generacio"] += 1
    estat.update({"x": AMPLE / 2, "meteorits": [], "punts": 0, "vides": 3,
                  "actiu": True, "ultim_meteorit": time.monotonic(), "vx": 0.0})
    dibuixar()
    estat_text.value = "<b>Partida en marxa</b> · Punts: 0 · Vides: 3"
    estat["tasca"] = asyncio.create_task(bucle_joc(estat["generacio"]))


def pausar(_=None):
    estat["generacio"] += 1
    estat["actiu"] = False
    tasca = estat.get("tasca")
    if tasca is not None and not tasca.done():
        tasca.cancel()
    dibuixar("PARTIDA PAUSADA")
    estat_text.value = "Partida pausada. Prem <b>Inicia / reinicia</b> per tornar a començar."

boto_iniciar = Button(description="Inicia / reinicia", button_style="success")
boto_pausar = Button(description="Pausa")
boto_iniciar.on_click(iniciar)
boto_pausar.on_click(pausar)
controls = VBox([canvas, HBox([boto_iniciar, boto_pausar]), estat_text])
display(controls)
dibuixar("Prem Inicia per jugar")

## Com funciona el joc

L’OVNI es mou automàticament. El controlador observa el meteorit que representa el perill més pròxim i calcula una velocitat lateral per evitar-lo. Prem **Pausa** per aturar la simulació o **Inicia / reinicia** per començar una partida nova.